# Multivariate Count and Filter

GitHub issue #36 asked for `.count()` to handle **multivariate** output, e.g.
estimating something like `P(X > 3, Y == 1, Z > 0)` for a joint simulation,
without having to write a custom function to pass into `.filter()`.

`count()` and `filter()` now accept one argument *per component* of a joint
outcome. Each argument can be:

- a **callable**, applied to that component only, e.g. `lambda x: x > 3`
- an **`(op, value)` tuple**, e.g. `('>', 3)`, using one of
  `'=='`, `'!='`, `'<'`, `'<='`, `'>'`, `'>='`
- **`None`**, to skip that component (no condition on it)

All of the per-component conditions are ANDed together. Passing a single
argument still works exactly as before (the univariate case).

In [ ]:
from symbulate import *

---
## Setting up a joint simulation

`X & Y & Z` below produces outcomes that are 3-tuples `(x, y, z)`, one
component per random variable.

In [ ]:
X, Y, Z = RV(Binomial(10, 0.5) * Binomial(5, 0.3) * Normal(0, 1))

sims = (X & Y & Z).sim(1000)
sims[:5]

0,"(4, 3, 0.4908923652469109)"
1,"(5, 2, -0.6491790733451659)"
2,"(4, 1, 0.4596910111052942)"
3,"(5, 1, -0.5057277338223459)"
4,"(4, 1, 0.8028353678950916)"
5,"(7, 4, -0.2820874991707055)"
6,"(5, 1, 0.10531026664209762)"
7,"(6, 0, -1.006832568240301)"
8,"(10, 1, -0.09274829621295513)"
...,...
999,"(5, 1, -1.553290898117368)"


---
## Before: writing a custom function

This is the workaround the issue mentions. It works, but it means writing
(and re-writing) a small function every time you want a different combination
of per-component conditions.

In [ ]:
def condition(outcome):
    x, y, z = outcome
    return x > 3 and y == 1 and z > 0

sims.filter(condition).count() 

157

---
## After: per-component conditions directly in `count()` / `filter()`

### Callables, one per component

Pass one callable per component, in order. Use `lambda o: True` (or `None`,
see below) for any component you don't want to filter on.

In [ ]:
sims.count(lambda x: x > 3, lambda y: y == 1, lambda z: z > 0)

157

### `(op, value)` tuples

For simple comparisons you don't even need a `lambda` — just pass the
operator as a string and the value to compare against.

In [ ]:
sims.count(('>', 3), ('==', 1), ('>', 0))

157

Both forms agree with the original, hand-written `condition()` function.

In [ ]:
assert sims.count(('>', 3), ('==', 1), ('>', 0)) == sims.filter(condition).count()
sims.count(('>', 3), ('==', 1), ('>', 0)) == sims.filter(condition).count()

True

### Skipping components with `None`

Use `None` in a component's position to leave it unconstrained. This is
useful when you only care about a subset of the components, e.g.
`P(Y == 1)` without touching `X` or `Z`.

In [ ]:
sims.count(None, ('==', 1), None)

373

In [ ]:
# Equivalent, but requires knowing the tuple layout ahead of time:
sims.count(lambda outcome: outcome[1] == 1)

373

### `filter()` supports the same syntax

`count()` is built on top of `filter()`, so every multivariate form above
also works with `filter()` directly &mdash; useful when you want the matching
outcomes themselves, not just how many there are.

In [ ]:
sims.filter(('>', 3), ('==', 1), ('>', 0))[:5]

0,"(4, 1, 0.4596910111052942)"
1,"(4, 1, 0.8028353678950916)"
2,"(5, 1, 0.10531026664209762)"
3,"(4, 1, 0.0924735010315413)"
4,"(6, 1, 2.0535143056613308)"
5,"(4, 1, 2.5728454282643445)"
6,"(4, 1, 0.3604125721349961)"
7,"(5, 1, 1.015474715650357)"
8,"(4, 1, 0.6027247726033305)"
...,...
156,"(4, 1, 0.6634990926796619)"


---
## Estimating a probability

Combining `count()` with the sample size gives a Monte Carlo estimate of a
joint probability like $P(X > 3, Y > 0, Z > 1)$, with no custom function
required.

In [ ]:
n = sims.count()  # total number of simulations
p_estimate = sims.count(('>', 3), ('>', 0), ('>', 1)) / n
p_estimate

0.124

---
## Mixing callables and tuples is not allowed

Each call to `count()`/`filter()` must use *either* callables *or*
`(op, value)` tuples for its non-`None` arguments &mdash; not both. Mixing
them raises a `TypeError` with a message showing the two supported forms.

In [ ]:
try:
    sims.count(lambda x: x > 3, ('==', 1))
except TypeError as e:
    print(e)

For multivariate filter/count, pass either:
  Per-component callables: count(lambda x: x > 3, lambda y: y == 1)
  Per-component (op, value) tuples: count(('>', 3), ('==', 1))
  Use None to skip a component: count(('>', 3), None, ('>', 0))


---
## Univariate `count()` is unchanged

Passing a single condition still behaves exactly as it always has &mdash;
the multivariate behavior only kicks in when more than one argument is
given.

In [ ]:
X = RV(Poisson(2))
x_sims = X.sim(1000)

x_sims.count(lambda x: x > 3), x_sims.count_gt(3)

(149, 149)